# Putting Example Difficulty in Practice with Early Exit

In the last notebook we explored the concept of example difficulty and prediction depth. How can we use this knowledge to our advantage? What kind of benefits can we see by leveraging our understanding of example difficulty.

In many growing domains of machine learning, we often will have systems constraints like inference latency, energy usage, etc.

As we noticed in the last homework, **some examples actually don't need to be passed through the entire network**. This concept was formalized as prediction depth.

So why do we need to pass the inputs through the entire network if they can be predicted correctly by an earlier layer?

Are there tradeoffs associated with not going through the entire network?

Much of this homework was inspired by the following paper:

https://arxiv.org/abs/1709.01686

## Concepts of BranchyNet

We will now have N exits, as we did with the KNN classifiers. However, now each exit will contribute to the loss in the following manner

$L_{\text{early exit}}(\hat{y}, y; \theta) = \sum_{i=1}^N w_i L(\hat{y_{\text{exit}}}, y; \theta)$

We will set the total loss of the network be a **weighted sum of the standard cross entropy losses at each exit**


# Baseline ResNet-18

To properly see the effects of Early Exit, let's set up a resnet without early exit as a baseline for computation and inference speed.

In [2]:
import torch
import torch.nn as nn
import copy
import torchvision
import torch.optim as optim
import torchvision.transforms as transforms
import sklearn
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy
from tqdm import tqdm
from architectures import EarlyExitResNet18
from copy import deepcopy

In [3]:
batch_size = 256

## MACS

MACS stands for multiply and accumulate - In the hardware, this corresponds to multiplying, then adding a number to an accumulator. We use MACS as a measurement of the amount of computation used.

In [4]:
!pip install torchprofile

In [5]:
import torchprofile

def get_model_macs(model, inputs) -> int:
    return torchprofile.profile_macs(model, inputs)

Let's set up our dataloader in the standard fashion. Similar to last homework, please download the data and put it in the same folder as this

In [6]:
data = np.load('data.npy', allow_pickle=True).item()
x_tensor = torch.FloatTensor(data['x'])
y_tensor = torch.LongTensor(data['y'])
dataset = torch.utils.data.TensorDataset(x_tensor, y_tensor)

test_data = np.load('test_data.npy', allow_pickle=True).item()
x_tensor = torch.FloatTensor(test_data['x'])
y_tensor = torch.LongTensor(test_data['y'])
test_dataset = torch.utils.data.TensorDataset(x_tensor, y_tensor)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


transform = transforms.Compose([
    transforms.ToPILImage(),                       # Convert arrays to PIL images
    transforms.Grayscale(num_output_channels=3),   # Convert grayscale to RGB
    transforms.Resize((224, 224)),                 # Resize all images to 224x224
    transforms.ToTensor(),                      # Convert the images to PyTorch tensors
])


resnet_dataset = deepcopy(dataset)
resnet_test_dataset = deepcopy(test_dataset)

resnet_dataset.transform = transform
resnet_trainloader = torch.utils.data.DataLoader(resnet_dataset, batch_size=batch_size, num_workers=2, shuffle=True)

resnet_test_dataset.transform = transform
resnet_testloader = torch.utils.data.DataLoader(resnet_test_dataset, batch_size=128, num_workers=2, shuffle=False)

In [7]:
resnet = EarlyExitResNet18()
resnet = resnet.to(device)

In [8]:
criterion = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet.parameters(), lr=0.0001)

# Training

Let's train a standard ResNet. Don't forget to pass in an entropy tolerance and set early_exit to False

In [9]:
step = 0
resnet_losses = []
for epoch in tqdm(range(10)):  # loop over the dataset multiple times
    running_loss = 0.0
    for i, data in enumerate(resnet_trainloader, 0):
        step += 1
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = inputs, labels = data[0].to(device), data[1].to(device)

        inputs = inputs.unsqueeze(1)
        inputs = inputs.repeat(1, 3, 1, 1)
        inputs = inputs.to(device)

        # zero the parameter gradients
        resnet_optimizer.zero_grad()


        # forward + backward + optimize
        # TODO
        outputs = resnet(inputs, 0, False)
        loss = criterion(outputs, labels)
        loss.backward()
        resnet_optimizer.step()
        resnet_losses.append(loss.item())
        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 50:.3f}')
            running_loss = 0.0



print('Finished Training')

  0%|          | 0/10 [00:00<?, ?it/s]

[1, 50] loss: 1.053
[1, 100] loss: 0.356


 10%|█         | 1/10 [00:05<00:52,  5.83s/it]

[2, 50] loss: 0.134
[2, 100] loss: 0.117


 20%|██        | 2/10 [00:09<00:38,  4.80s/it]

...

[9, 50] loss: 0.009
[9, 100] loss: 0.013


 90%|█████████ | 9/10 [00:38<00:04,  4.14s/it]

[10, 50] loss: 0.006
[10, 100] loss: 0.013


100%|██████████| 10/10 [00:42<00:00,  4.27s/it]

Finished Training


### Fill in the code below to evaluate the ResNet

In [10]:
resnet.eval()
total_macs = 0

for epoch in range(1):  # loop over the dataset multiple times

    total_correct = 0
    with torch.no_grad():
        for i, data in tqdm(enumerate(resnet_testloader, 0)):
            # get the inputs; data is a list of [inputs, labels]

            inputs, labels = inputs, labels = data[0].to(device), data[1].to(device)

            inputs = inputs.unsqueeze(1)
            inputs = inputs.repeat(1, 3, 1, 1)
            inputs = inputs.to(device)
            # forward + backward + optimize
            # TODO
            outputs = resnet(inputs, 0, False)
            total_macs += get_model_macs(resnet, (inputs, 0, False))


            indices = torch.argmax(outputs, dim=1)

            total_correct += torch.sum(labels == indices)


print(f'Accuracy: {total_correct/6000} %')
print('Total MACS: ', total_macs)

47it [00:05,  9.00it/s]

Accuracy: 0.8956666588783264 %
Total MACS:  3336213504000


What was the accuracy with a regular ResNet-18?

Inference Speed?

Total MACS?

In [11]:
resnet_early = EarlyExitResNet18()
resnet_early = resnet.to(device)
criterion = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet.parameters(), lr=0.0001)

### Fill in the code below to train the early exit network

In [12]:
step = 0
resnet_losses = []
for epoch in tqdm(range(10)):  # loop over the dataset multiple times
    running_loss = 0.0
    for i, data in enumerate(resnet_trainloader, 0):
        step += 1
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = inputs, labels = data[0].to(device), data[1].to(device)

        inputs = inputs.unsqueeze(1)
        inputs = inputs.repeat(1, 3, 1, 1)
        inputs = inputs.to(device)

        # zero the parameter gradients
        resnet_optimizer.zero_grad()


        # forward + backward + optimize
        # TODO Use w_0 = 1 and w_i = 0.3 for i > 0
        outputs = resnet_early.early_train(inputs)
        loss = 0.3 * criterion(outputs[0], labels) + 0.3 * criterion(outputs[1], labels) + 0.3 * criterion(outputs[2], labels) + 1.0 * criterion(outputs[3], labels)
        loss.backward()
        resnet_optimizer.step()
        resnet_losses.append(loss.item())
        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 50:.3f}')
            running_loss = 0.0



print('Finished Training')

  0%|          | 0/10 [00:00<?, ?it/s]

[1, 50] loss: 3.536
[1, 100] loss: 0.803


 10%|█         | 1/10 [00:04<00:39,  4.40s/it]

[2, 50] loss: 0.520
[2, 100] loss: 0.452


 20%|██        | 2/10 [00:08<00:34,  4.30s/it]

...

[9, 50] loss: 0.085
[9, 100] loss: 0.102


 90%|█████████ | 9/10 [00:38<00:04,  4.25s/it]

[10, 50] loss: 0.059
[10, 100] loss: 0.077


100%|██████████| 10/10 [00:42<00:00,  4.26s/it]

Finished Training


### Fill in the code below to evaluate the early exit network

In [13]:
resnet_early.eval()
exiting_points = {0:0, 1:0, 2:0, 3:0}
entropies = []
total_early_macs = 0

for epoch in range(1):  # loop over the dataset multiple times

    total_early_correct = 0
    with torch.no_grad():
        for i, data in tqdm(enumerate(resnet_testloader, 0)):
            # get the inputs; data is a list of [inputs, labels]

            inputs, labels = inputs, labels = data[0].to(device), data[1].to(device)

            inputs = inputs.unsqueeze(1)
            inputs = inputs.repeat(1, 3, 1, 1)
            inputs = inputs.to(device)
            # forward + backward + optimize
            # TODO Use an entropy tolerance of 0.05
            outputs, num, curr_entropy = resnet_early(inputs, 0.05, True)
            entropies.append(curr_entropy)
            total_early_macs += get_model_macs(resnet_early, (inputs, 0.05))


            exiting_points[num] += 1


            indices = torch.argmax(outputs, dim=1)

            total_early_correct += torch.sum(labels == indices)


print(f'Accuracy: {total_correct/6000} %')
print('Num Exiting: ', exiting_points)
print('Total MACS: ', total_early_macs)
entropies = sorted(entropies)
print(len(entropies))
print('Entropies: ', entropies[0], entropies[len(entropies)//4], entropies[len(entropies)//2], entropies[3*len(entropies)//4], entropies[-1])

47it [00:04, 10.81it/s]

Accuracy: 0.8956666588783264 %
Num Exiting:  {0: 0, 1: 20, 2: 3, 3: 24}
Total MACS:  2597679341568
47
Entropies:  0.005835264 0.020299572 0.03907223 0.05752672 0.098075286


What was the accuracy with an early exit ResNet-18?

Inference Speed?

Total MACS?

In [14]:
print(f'Ratio of Standard to Early Exit MACS: {total_macs/total_early_macs}')

Ratio of Standard to Early Exit MACS: 1.28430536079415


## Entropy

Play around with the entropy tolerance to see how low you can get the MACS while keeping 90 percent or greater accuracy.

How did early exit do? Compare accuracy and MACS.

## Open Question

No solutions will be provided for this question:

When would we use early exit, versus just using a smaller model? What factors should we consider?

How does early exit relate to example difficulty?
